# AndinaLog 03B | Warehouse Costs | Diagnóstico v2

Conserva las seis filas Bronze y explica cada anomalía por campo.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ENTORNO='auto';RUTA_PROYECTO_DRIVE='/content/drive/MyDrive/GIAD'
COLUMNAS_BRONZE=['centro_distribucion','rotacion_stock_dias','perdida_mermas_bob','periodo_mes','costo_almacenamiento_mensual_bob']
CENTROS={'Cochabamba','La Paz','Santa Cruz','Oruro','Tarija'}
def encontrar_raiz():
    if ENTORNO=='drive' or (ENTORNO=='auto' and 'google.colab' in sys.modules):
        from google.colab import drive;drive.mount('/content/drive');return Path(RUTA_PROYECTO_DRIVE)
    for p in [Path.cwd(),*Path.cwd().parents]:
        if (p/'datasets/AndinaLog_03B_Bronce/andinalog_warehouse_costs.csv').is_file():return p
    raise FileNotFoundError('No se encontró la raíz')
RAIZ=encontrar_raiz();RUTA_BRONZE=RAIZ/'datasets/AndinaLog_03B_Bronce/andinalog_warehouse_costs.csv'
SALIDAS=RAIZ/'proyecto-integrador/01_diagnostico/andinalog_warehouse_costs/salidas'
bronze=pd.read_csv(RUTA_BRONZE,dtype='string',encoding='utf-8-sig',keep_default_na=False);assert list(bronze.columns)==COLUMNAS_BRONZE
df=bronze.copy();df.insert(0,'fila_bronze',range(1,len(df)+1))
for x in COLUMNAS_BRONZE:df[f'{x}_en_cuarentena']=False;df[f'{x}_motivo']=''
def marcar(x,m,motivo,cuarentena=True):
 m=pd.Series(m,index=df.index).fillna(False).astype(bool)
 if cuarentena:df.loc[m,f'{x}_en_cuarentena']=True
 p=df.loc[m,f'{x}_motivo'];df.loc[m,f'{x}_motivo']=p.where(p.eq(''),p+'; ')+motivo
print('Filas Bronze:',len(df))


## Reglas de calidad

El periodo es mensual, sin zona horaria. Los importes están en BOB y la clave natural es centro más periodo.


In [ ]:
for x in COLUMNAS_BRONZE:marcar(x,df[x].str.strip().eq(''),'Valor faltante')
marcar('centro_distribucion',~df.centro_distribucion.str.strip().isin(CENTROS),'Centro fuera del catálogo')
periodo=pd.to_datetime(df.periodo_mes,format='%Y-%m',errors='coerce');marcar('periodo_mes',periodo.isna(),'Formato esperado AAAA-MM')
num={x:pd.to_numeric(df[x].str.strip(),errors='coerce') for x in ['rotacion_stock_dias','perdida_mermas_bob','costo_almacenamiento_mensual_bob']}
for x,n in num.items():marcar(x,df[x].str.strip().ne('')&n.isna(),'Valor no numérico')
marcar('rotacion_stock_dias',num['rotacion_stock_dias'].notna()&num['rotacion_stock_dias'].le(0),'Rotación debe ser positiva')
for x in ['perdida_mermas_bob','costo_almacenamiento_mensual_bob']:marcar(x,num[x].notna()&num[x].lt(0),'Monto no puede ser negativo')
clave=df.centro_distribucion.str.strip()+'|'+df.periodo_mes.str.strip();repetida=clave.duplicated(False);copia_incompleta=clave.duplicated()&df.rotacion_stock_dias.str.strip().eq('')
for x in COLUMNAS_BRONZE:marcar(x,copia_incompleta,'Copia incompleta posterior de la misma clave centro-periodo',x in ['centro_distribucion','rotacion_stock_dias','periodo_mes'])


## Integridad referencial y exportación

Los centros se contrastan con Inventario y Flota Silver. No se reemplazan montos contables mediante cálculos derivados.


In [ ]:
ri=RAIZ/'proyecto-integrador/02_tratamiento/andinalog_inventory_tracking/salidas/andinalog_inventory_tracking_didactico_v2_silver.csv';rf=RAIZ/'proyecto-integrador/02_tratamiento/andinalog_flota/salidas/andinalog_flota_didactico_v2_silver.csv'
inv=pd.read_csv(ri,dtype='string',encoding='utf-8-sig');flo=pd.read_csv(rf,dtype='string',encoding='utf-8-sig');refs=set(inv.centro_distribucion)|set(flo.centro_distribucion_base_tratado);marcar('centro_distribucion',~df.centro_distribucion.isin(refs),'Centro sin referencia Silver')
df['en_cuarentena']=df[[f'{x}_en_cuarentena' for x in COLUMNAS_BRONZE]].any(axis=1);public=['fila_bronze',*COLUMNAS_BRONZE,*[y for x in COLUMNAS_BRONZE for y in (f'{x}_en_cuarentena',f'{x}_motivo')],'en_cuarentena'];diagnosticado=df[public].copy();cuarentena=diagnosticado.loc[diagnosticado.en_cuarentena].copy();metricas={'filas_bronze':len(bronze),'filas_diagnosticadas':len(diagnosticado),'filas_cuarentena':len(cuarentena),'claves_repetidas':int(repetida.sum()),'copias_incompletas_posteriores':int(copia_incompleta.sum())};[metricas.update({f'cuarentena_{x}':int(df[f'{x}_en_cuarentena'].sum())}) for x in COLUMNAS_BRONZE];reporte=pd.DataFrame([{'metrica':k,'valor':v} for k,v in metricas.items()]);pd.testing.assert_frame_equal(diagnosticado[COLUMNAS_BRONZE],bronze);assert len(cuarentena)==1
SALIDAS.mkdir(parents=True,exist_ok=True);b='andinalog_warehouse_costs_didactico_v2_';diagnosticado.to_csv(SALIDAS/(b+'diagnosticado.csv'),index=False,encoding='utf-8-sig');cuarentena.to_csv(SALIDAS/(b+'cuarentena.csv'),index=False,encoding='utf-8-sig');reporte.to_csv(SALIDAS/(b+'reporte_calidad.csv'),index=False,encoding='utf-8-sig');print(metricas);display(diagnosticado)
